# Two-bay, two-storey RC frame

OPENSTRUCTURE-ANALYSIS v0.1. The frame lives in `two_bay_two_storey.yaml` (bays of 6 m, storeys of 3.5 m). This notebook loads that file, runs the linear-static analysis, and designs one beam and one column to both วสท. 1008 and ACI 318-19.

โครงข้อแข็ง 2 ช่วง 2 ชั้น วิเคราะห์แรงสถิตเชิงเส้น แล้วออกแบบคานชั้นล่างและเสาตอม่อตามวิธีกำลัง ทั้งสองมาตรฐาน รายงานแสดงทีละขั้น

Global y is upward. Moments that compress the local +y face are positive, so sagging is positive on a beam drawn left to right.

In [ ]:
from pathlib import Path

from openstructure.analysis import LinearStaticEngine
from openstructure.io import load_model
from openstructure.pipeline import run_file
from openstructure.plotting import plot_member_forces

example = Path("two_bay_two_storey.yaml")
if not example.exists():
    example = Path("examples/two_bay_two_storey.yaml")

model, raw = load_model(example)
print(model.title)
print(f"{len(model.nodes)} nodes, {len(model.members)} members, {len(model.combinations)} combinations")

In [ ]:
engine = LinearStaticEngine()
result = engine.analyze(model)
gravity = result.by_combination["EIT-1.4D+1.7L"]
beam = gravity.members["B10"]
sagging = max(beam.stations, key=lambda station: station.m_kNm)
hogging = min(beam.stations, key=lambda station: station.m_kNm)
print(f"B10 sagging  {sagging.m_kNm:.2f} kN·m at x = {sagging.x_m:.2f} m")
print(f"B10 hogging  {hogging.m_kNm:.2f} kN·m at x = {hogging.x_m:.2f} m")
fx, fy, mz = gravity.equilibrium_kN
print(f"Equilibrium residual  Fx = {fx:.3g} kN, Fy = {fy:.3g} kN, M = {mz:.3g} kN·m")

In [ ]:
figure = plot_member_forces("B10", beam, "EIT-1.4D+1.7L")
figure

## Member design

The pipeline reads Mu, Vu, and Pu from the analysis. Beam shear is taken at `face_offset_mm + d` from the node, which is the column face plus d on this centre-line model. Hogging Mu is the moment at the node, not the smaller moment at the column face. The full step-by-step report is in `report.md` when you run the command line; here the governing line of each check is printed.

Both codes return the same tension steel for this beam, because the section is singly reinforced and both use φ = 0.90. They do not share φ for shear or for a tied column. See `docs/CODE_ASSUMPTIONS.md`.

In [ ]:
output = run_file(example, out_dir=None, plots=False)
for packet in output.designs:
    print(f"{packet.kind:14} {packet.code_id:10} {packet.status:6} {packet.summary}")

In [ ]:
flexure = next(
    packet for packet in output.designs
    if packet.kind == "beam_flexure" and packet.code_id == "EIT1008"
)
# The markdown report starts with the summary table, then the governing calculation.
text = flexure.report_markdown
start = text.find("# RC beam flexure")
print(text[start:start + 1800])